In [124]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [125]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [126]:
# query = """
# SELECT DISTINCT v.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-08-06'
#     AND v.Fecha <= '2025-08-11'
#     AND v.Venta_Neta > 0
#     AND Contacto LIKE '%Cel%'
# GROUP BY  v.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query, engine)
# df_ventas.head()

In [127]:
# #consulta segmentada 
# query_com = """
# SELECT DISTINCT v.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-07-25'
#     AND v.Fecha <= '2025-07-30'
#     AND v.Venta_Neta > 0
#     AND Sku= '29VC240A23'
# 	AND Canal IN ('Shopify','CHATCENTER WEB')
# GROUP BY  v.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """
# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_com, engine)
# df_ventas.head()

In [128]:
# CONSULTAS CUMPLEAÑOS

query_data = """
SELECT DISTINCT v.Cliente, 
	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
	cc.Email, cc.Celular, 
	v.Fecha AS Fecha_Venta 
	,SUM(Venta_Neta) AS Venta, 
	Canal
FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON cc.Cliente = v.Cliente
WHERE Codigo_Descuento IN ('0310 Clientes Cumple Julio Reactivo 25%','0309 Clientes Cumple Julio 2 Meses 25%','CUMPLE0825')
	AND Fecha BETWEEN '2025-08-09' AND '2025-08-28'
	AND NOT EXISTS (SELECT 1 FROM COMERSSIA.PROVENZAL.dbo.EmpleadosProvenzal e WHERE e.Codigo = v.cliente ) 
GROUP BY  v.Cliente,
       cc.Nombres + ' ' +  cc.Apellidos,
       cc.Email,
       cc.Celular,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_data, engine)
df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Venta,Canal
0,C1000089807,PAULINA BERNAL POSADA,PAULINABERNALPOSADA@HOTMAIL.COM,3127739432,2025-08-12,34033.61,Retail
1,C1000593985,JULIANA ROMERO,JULI.ROMERO13@HOTMAIL.COM,3106247450,2025-08-09,116596.64,Retail
2,C10029397,JULIAN ESTRADA,jujuestlondono@gmail.com,3166213281,2025-08-09,432754.50,Shopify
3,C1005770920,VANESSA BELTRAN,BELTRANVANESSA345@GMAIL.COM,3057877616,2025-08-27,187815.13,Retail
4,C1010172352,DIANA MARCELA GALINDO GUTIERREZ,DIMARG12@HORTMAIL.COM,3006783314,2025-08-25,351680.68,Retail


In [129]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['phone', 'campaignName', 'sentAt',  'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'phone': 'Celular',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'clicks': 'clics'
}, inplace=True)

# Extraer primera fecha de apertura (si viene como lista en string)
df_campania['clics'] = df_campania['clics'].apply(
    lambda x: eval(x)[0] if isinstance(x, str) and x.startswith('[') and len(eval(x)) > 0 else None
)
df_campania['clics'] = pd.to_datetime(df_campania['clics'], errors='coerce').dt.tz_localize(None)
df_campania['fecha_envio'] = pd.to_datetime(df_campania['fecha_envio'], errors='coerce').dt.tz_localize(None).dt.date

df_enviados = df_campania[df_campania['fecha_envio'].notnull()].copy()
df_clics =  df_campania[df_campania['clics'].notnull()].copy()

fecha_maxima = pd.to_datetime('2025-08-28').date()
df_clics_filtrados = df_clics[df_clics['clics'].dt.date <= fecha_maxima]

df_clics_filtrados = df_clics_filtrados.drop_duplicates(subset='Celular', keep='first')

total_clics = df_clics_filtrados['Celular'].nunique()


In [130]:
df_enviados['Celular'] = df_enviados['Celular'].astype(str)
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# Filtrar ventas dentro de 5 días desde la fecha de envio
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['fecha_envio']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 30)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que hicieron clic hasta {fecha_maxima}: {total_clics}")
print(f"Clientes Efectivos: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.0f}")
# Resultado final
df_atribucion.head()

Número de clientes que hicieron clic hasta 2025-08-28: 1792
Clientes Efectivos: 148
Total de venta atribuida: 41,619,615


,Cliente,Nombre,Email,Celular,Fecha_Venta,Venta,Canal,campania,fecha_envio,clics,dias_post_apertura
0,C1000089807,PAULINA BERNAL POSADA,PAULINABERNALPOSADA@HOTMAIL.COM,3127739432,2025-08-12,34033.61,Retail,Reactivo CumpleaÃ±os Julio,2025-08-09,NaT,3
1,C1000593985,JULIANA ROMERO,JULI.ROMERO13@HOTMAIL.COM,3106247450,2025-08-09,116596.64,Retail,Reactivo CumpleaÃ±os Julio,2025-08-09,NaT,0
2,C10029397,JULIAN ESTRADA,jujuestlondono@gmail.com,3166213281,2025-08-09,432754.50,Shopify,Reactivo CumpleaÃ±os Julio,2025-08-09,2025-08-09 16:40:21,0
3,C1005770920,VANESSA BELTRAN,BELTRANVANESSA345@GMAIL.COM,3057877616,2025-08-27,187815.13,Retail,Reactivo CumpleaÃ±os Julio,2025-08-09,NaT,18
4,C1010172352,DIANA MARCELA GALINDO GUTIERREZ,DIMARG12@HORTMAIL.COM,3006783314,2025-08-25,351680.68,Retail,Reactivo CumpleaÃ±os Julio,2025-08-09,NaT,16


In [131]:
df_atribucion.to_excel('Ventas.xlsx', index=False)